# Demonstrasi: Visualisasi & Cerita Data — PIF304 Pengantar Data Sains
**Pertemuan 7 — Sub-CPMK S03: Memilih visualisasi dan menyusun cerita data**

Grafik yang baik menjawab **satu pertanyaan** dengan jujur dan mudah dibaca. Urutan kerja kita:

1. Tentukan **pertanyaan** → pilih **bentuk** grafik
2. Gambar versi bawaan → perbaiki (*makeover*)
3. Kenali grafik yang **menyesatkan**
4. Warna dan aksesibilitas
5. Susun **cerita data**: konteks → temuan → implikasi dan batas

**Dataset:** `Data_Survei_Bersih_PDS.xlsx` (45 responden; `jarak_km_bersih` hasil keputusan Pertemuan 5).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

### Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih **`Data_Survei_Bersih_PDS.xlsx`** yang sudah Anda unduh dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Silakan pilih file: Data_Survei_Bersih_PDS.xlsx")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Google Colab (mis. Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Survei_Bersih_PDS.xlsx"

df = pd.read_excel(nama_file)   # sheet pertama = "Data"
print("File:", nama_file, "| ukuran:", df.shape)
df.head()

In [ ]:
# Gaya dasar: latar bersih, garis bantu tipis, teks gelap
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "axes.edgecolor": "#C3C7CC", "grid.color": "#E6E8EB", "grid.linewidth": 0.8,
})
urut_ai = ["Kadang-kadang", "Sering", "Sangat sering"]

## 1. Pertanyaan Menentukan Bentuk Grafik
| Pertanyaan | Tugas data | Bentuk yang cocok |
|---|---|---|
| Kategori mana yang paling banyak? | perbandingan | diagram batang (diurutkan) |
| Bagaimana sebaran nilainya? | distribusi | histogram, boxplot |
| Apakah dua angka berhubungan? | hubungan | scatter plot |
| Bagaimana komposisi tiap kelompok? | bagian dari keseluruhan | batang bertumpuk 100% |
| Bagaimana perubahan dari waktu ke waktu? | tren | diagram garis |

### 1a. Perbandingan → batang diurutkan
*Pertanyaan: waktu belajar mana yang paling banyak dipilih?*

In [ ]:
urutan = df["waktu_belajar_favorit"].value_counts()
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.barh(urutan.index[::-1], urutan.values[::-1], color=TEAL, height=0.6)
for y, v in enumerate(urutan.values[::-1]):
    ax.text(v + 0.5, y, f"{v} ({v/len(df):.0%})", va="center", fontsize=10)
ax.set(title="Malam adalah waktu belajar favorit 29 dari 45 responden", xlabel="jumlah responden", xlim=(0, 36))
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

Tiga keputusan kecil yang membuat grafik ini mudah dibaca: batang **diurutkan**, nilai **ditulis langsung** di ujung batang, dan **judul menyatakan temuan** (bukan sekadar "Waktu Belajar").

### 1b. Distribusi → histogram dan boxplot
*Pertanyaan: berapa lama perjalanan responden ke kampus, dan apakah berbeda menurut moda?*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].hist(df["waktu_tempuh_menit"], bins=range(0, 30, 3), color=TEAL, edgecolor="white", linewidth=1.5)
axes[0].set(title="Sebagian besar responden tiba dalam 10 menit", xlabel="waktu tempuh (menit)", ylabel="jumlah responden")

dua_moda = df[df["moda_transportasi"].isin(["Jalan kaki", "Sepeda motor"])]
sns.boxplot(data=dua_moda, x="waktu_tempuh_menit", y="moda_transportasi", color=TEAL, width=0.5, ax=axes[1])
axes[1].set(title="Waktu tempuh per moda (ojek online n=1 tidak ditampilkan)", xlabel="waktu tempuh (menit)", ylabel="")
plt.tight_layout(); plt.show()

### 1c. Hubungan → scatter plot
*Pertanyaan: apakah jarak berkaitan dengan waktu tempuh?*

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.8))
sns.scatterplot(data=dua_moda, x="jarak_km_bersih", y="waktu_tempuh_menit", hue="moda_transportasi", style="moda_transportasi",
                palette=PALET[:2], s=70, edgecolor="white", linewidth=1, ax=ax)
ax.set(title="Makin jauh, makin lama - dan polanya berbeda per moda", xlabel="jarak ke kampus (km)", ylabel="waktu tempuh (menit)")
ax.legend(title="", frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

Identitas moda disandikan dengan **warna dan bentuk penanda sekaligus**, sehingga tetap terbaca oleh pembaca buta warna atau saat dicetak hitam-putih.

### 1d. Komposisi → batang bertumpuk 100%
*Pertanyaan: apakah frekuensi pemakaian AI berbeda antara Semester 3 dan 5?*

In [ ]:
komposisi = pd.crosstab(df["semester"], df["frekuensi_pakai_ai"], normalize="index")[urut_ai] * 100
jumlah = pd.crosstab(df["semester"], df["frekuensi_pakai_ai"])[urut_ai]
urutan_warna = ["#A9DCE4", "#4FB4C4", "#0E6E7E"]      # satu rona, terang -> gelap, karena kategorinya BERURUTAN

fig, ax = plt.subplots(figsize=(8, 3.0))
kiri = np.zeros(len(komposisi))
for kat, warna in zip(urut_ai, urutan_warna):
    ax.barh(komposisi.index, komposisi[kat], left=kiri, color=warna, edgecolor="white", linewidth=2, height=0.6, label=kat)
    for i, (p, k) in enumerate(zip(komposisi[kat], jumlah[kat])):
        ax.text(kiri[i] + p / 2, i, f"{p:.0f}% ({k})", ha="center", va="center", fontsize=9,
                color="white" if warna == "#0E6E7E" else "#13233F")
    kiri += komposisi[kat].to_numpy()
ax.set(title="'Sangat sering' memakai AI: 48% di Semester 5, 12% di Semester 3", xlabel="% responden per semester (jumlah)", xlim=(0, 100))
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
fig.legend(ncol=3, frameon=False, loc="lower center")
plt.tight_layout(rect=(0, 0.1, 1, 1)); plt.show()

## 2. Makeover: dari Bawaan ke Bermakna
Grafik bawaan pustaka jarang langsung layak tampil. Bandingkan dua versi untuk pertanyaan yang sama: *platform belajar apa yang paling banyak dipakai?*

In [ ]:
plat = df["platform_belajar"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

# SEBELUM: pie bawaan, 6 irisan, warna pelangi, tanpa pesan
axes[0].pie(plat.values, labels=plat.index, colors=plt.cm.rainbow(np.linspace(0, 1, len(plat))), textprops={"fontsize": 8})
axes[0].set_title("SEBELUM: platform_belajar", loc="center")

# SESUDAH: batang diurutkan, satu warna latar, satu sorotan, label langsung, judul = temuan
warna = [ORANGE if k == plat.index[0] else GREY for k in plat.index[::-1]]
axes[1].barh(plat.index[::-1], plat.values[::-1], color=warna, height=0.65)
for y, v in enumerate(plat.values[::-1]):
    axes[1].text(v + 0.5, y, str(v), va="center", fontsize=10)
axes[1].set(title="SESUDAH: 3 dari 4 responden belajar terutama dengan AI", xlabel="jumlah responden (n = 45)", xlim=(0, 40))
axes[1].grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

Yang diubah dan alasannya:

| Perubahan | Alasan |
|---|---|
| Pie → batang | Mata membandingkan **panjang** jauh lebih akurat daripada **sudut**; irisan kecil tak terbaca |
| Diurutkan | Peringkat langsung terlihat |
| Pelangi → abu-abu + satu sorotan | Warna hanya dipakai untuk hal yang ingin ditunjukkan |
| Angka langsung di batang | Pembaca tidak perlu menebak dari sumbu |
| Judul = temuan | Pembaca tahu apa yang harus dilihat |

## 3. Grafik yang Menyesatkan
### 3a. Sumbu terpotong
Rata-rata `tingkat_ketertarikan`: Semester 3 = 4,00; Semester 5 = 3,62.

In [ ]:
rata = df.groupby("semester")["tingkat_ketertarikan"].mean()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, batas, judul in [(axes[0], (3.5, 4.05), "MENYESATKAN: sumbu mulai dari 3,5"), (axes[1], (0, 5), "JUJUR: sumbu mulai dari 0 (skala 1-5)")]:
    ax.bar(rata.index, rata.values, color=TEAL, width=0.5)
    for x, v in enumerate(rata.values):
        ax.text(x, v + (batas[1] - batas[0]) * 0.02, f"{v:.2f}", ha="center", fontsize=10)
    ax.set(ylim=batas, title=judul, ylabel="rata-rata ketertarikan"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

Data yang sama, kesan yang berlawanan. Pada grafik kiri, batang Semester 3 tampak **empat kali** lebih tinggi; kenyataannya selisihnya 0,38 poin pada skala 1–5.

> **Aturan:** diagram batang menyandikan nilai dengan *panjang*, maka sumbunya **harus mulai dari nol**. (Diagram garis dan scatter boleh tidak mulai dari nol, asalkan sumbunya diberi label jelas.)

### 3b. Bentuk lain yang perlu diwaspadai
- **Pie/3D** dengan banyak irisan atau efek perspektif
- **Dua sumbu-y** pada satu grafik: skala kiri-kanan bisa diatur agar dua garis tampak "berhubungan"
- **Memilih-milih rentang** (*cherry-picking*) agar tren tampak naik/turun
- **Persentase tanpa n**: "100% pengguna ojek online tertarik Data Sains" (n = 1)
- **Rata-rata tanpa sebaran**

## 4. Warna dan Aksesibilitas
Warna punya **pekerjaan**. Pilih jenis palet sesuai pekerjaannya:

| Pekerjaan warna | Jenis palet | Contoh di notebook ini |
|---|---|---|
| Membedakan kategori tanpa urutan | **kategorikal** (rona berbeda) | moda transportasi |
| Menunjukkan urutan/besaran | **sekuensial** (satu rona, terang → gelap) | frekuensi pakai AI |
| Menunjukkan dua arah dari titik tengah | **divergen** (dua rona + netral) | selisih dari rata-rata |
| Menyorot satu hal | abu-abu + **satu aksen** | platform AI |

Daftar periksa aksesibilitas:
- Jangan mengandalkan **warna saja** → tambahkan label langsung, bentuk penanda, atau pola.
- Hindari pasangan **merah–hijau** (sulit dibedakan oleh ±8% laki-laki).
- Kontras teks cukup; ukuran huruf ≥ 9–10 pt pada gambar akhir.
- Sertakan **teks alternatif**: satu kalimat yang menyatakan isi grafik bagi pembaca yang tidak dapat melihatnya.

In [ ]:
# Simulasi sederhana: bagaimana grafik terlihat tanpa warna (cetak hitam-putih)?
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
sns.scatterplot(data=dua_moda, x="jarak_km_bersih", y="waktu_tempuh_menit", hue="moda_transportasi", palette=["#777777", "#777777"], s=60, legend=False, ax=axes[0])
axes[0].set(title="Hanya warna -> identitas hilang saat dicetak abu-abu", xlabel="jarak (km)", ylabel="waktu tempuh (menit)")
sns.scatterplot(data=dua_moda, x="jarak_km_bersih", y="waktu_tempuh_menit", style="moda_transportasi", color="#777777", s=60, ax=axes[1])
axes[1].set(title="Warna + bentuk -> tetap terbaca", xlabel="jarak (km)", ylabel="waktu tempuh (menit)"); axes[1].legend(title="", frameon=False)
plt.tight_layout(); plt.show()

## 5. Cerita Data
Grafik menjawab "apa". Cerita data menambahkan "lalu apa?" dengan tiga bagian:

1. **Konteks** — siapa yang bertanya dan keputusan apa yang dihadapi
2. **Temuan** — 1–3 grafik, masing-masing satu pesan, dengan anotasi
3. **Implikasi dan batas** — apa yang disarankan, dan apa yang tidak boleh disimpulkan

Contoh untuk skenario kelas: *dosen ingin menentukan slot konsultasi daring.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 1.25]})

# Panel 1: temuan utama
warna = [ORANGE if k == "Malam" else GREY for k in urutan.index[::-1]]
axes[0].barh(urutan.index[::-1], urutan.values[::-1], color=warna, height=0.6)
for y, v in enumerate(urutan.values[::-1]):
    axes[0].text(v + 0.5, y, f"{v/len(df):.0%}", va="center", fontsize=10)
axes[0].set(title="1  Malam dipilih 64% responden", xlabel="jumlah responden", xlim=(0, 36)); axes[0].grid(axis="y", visible=False)

# Panel 2: apakah temuan itu berlaku di kedua semester?
per_sem = pd.crosstab(df["waktu_belajar_favorit"], df["semester"], normalize="columns").reindex(urutan.index) * 100
x = np.arange(len(per_sem)); lebar = 0.36
for i, (sem, w) in enumerate(zip(per_sem.columns, PALET[:2])):
    axes[1].bar(x + (i - 0.5) * (lebar + 0.03), per_sem[sem], width=lebar, color=w, label=sem)
    for xi, v in zip(x, per_sem[sem]):
        axes[1].text(xi + (i - 0.5) * (lebar + 0.03), v + 1.5, f"{v:.0f}%", ha="center", fontsize=9)
axes[1].set_xticks(x, per_sem.index)
axes[1].set(title="2  Polanya serupa di Semester 3 dan Semester 5", ylabel="% responden per semester", ylim=(0, 80))
axes[1].legend(frameon=False); axes[1].grid(axis="x", visible=False)

fig.suptitle("Kapan mahasiswa belajar? Dasar pemilihan slot konsultasi daring", x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("cerita_data_demo_P7.png", dpi=200, bbox_inches="tight"); plt.show()

**Narasi (±90 kata):**

> **Konteks.** Dosen pengampu ingin membuka sesi konsultasi daring tambahan dan perlu memilih waktunya.
> **Temuan.** Dari 45 responden survei, 29 orang (64%) memilih malam sebagai waktu belajar favorit, dan polanya serupa di Semester 3 (67%) maupun Semester 5 (62%).
> **Implikasi.** Slot malam paling mungkin sesuai dengan kebiasaan belajar responden.
> **Batas.** Survei diisi secara sukarela oleh 45 mahasiswa sehingga belum tentu mewakili seluruh mahasiswa; "malam" juga belum menyebut jam tertentu. Disarankan survei lanjutan singkat tentang jam yang diinginkan sebelum jadwal ditetapkan.

**Teks alternatif:** *Dua diagram batang. Kiri: malam dipilih 29 dari 45 responden (64%), pagi 11, siang 5. Kanan: persentase per semester hampir sama di Semester 3 dan Semester 5.*

## Ringkasan: Daftar Periksa Sebelum Menampilkan Grafik
- [ ] Grafik menjawab **satu** pertanyaan, dan bentuknya sesuai tugas data
- [ ] Judul menyatakan **temuan**; sumbu berlabel dan bersatuan
- [ ] Batang mulai dari **nol**; tidak ada dua sumbu-y
- [ ] Warna punya pekerjaan; identitas tidak bergantung pada warna saja
- [ ] Persentase disertai **n**
- [ ] Ada kalimat **batas interpretasi**
- [ ] Gambar disimpan dengan resolusi cukup (`dpi=200`) dan punya teks alternatif

**Berikutnya (Pertemuan 8):** Ujian Tengah Semester — konsep, problem framing, mutu dan etika data, statistik deskriptif, pembersihan, transformasi, dan visualisasi.